In [1]:
import pandas as pd
import requests
import time

In [16]:
df = pd.read_csv("data_2026-03-02_2026-03-08.csv")
# Filter out bad coordinates
df = df.dropna(subset = ['longitude', 'latitude'])
df = df[(df['longitude'] != 0) & (df['latitude'] != 0)]

# Clean and combine address data for later processing
df['street_name'] = df['street_name'].apply(lambda x : x.title())
df['address'] = df.apply(lambda x : str(x['house_number']) + ' ' + x['street_name'] ,axis = 1)

df = df.iloc[6:10]

df = df[
    ['inspection_type',
     'job_id',
     'job_progress',
     'address',
     'zip_code',
     'latitude',
     'longitude',
     'result',
     'approved_date',
     'nta'
     ]
]
df.head()

,inspection_type,job_id,job_progress,address,zip_code,latitude,longitude,result,approved_date,nta
7,Compliance,PC8654073,2,447 East 9 Street,10009.0,40.727378,-73.982934,Rat Activity,2026-03-03T09:20:43.000,East Village
8,Initial,PC8663068,1,762 Putnam Avenue,11221.0,40.686179,-73.931268,Rat Activity,2026-03-03T10:16:20.000,Bedford-Stuyvesant (East)
9,Initial,PC8672869,1,87-82 160 Street,11432.0,40.707470,-73.801622,Failed for Rat Act,2026-03-04T11:52:43.000,Jamaica
10,Initial,PC8658711,1,1479 Dahill Road,11204.0,40.611951,-73.974637,Failed for Rat Act,2026-03-04T14:23:27.000,Mapleton-Midwood (West)


In [23]:
# Radius in metres to search around the geocoded point
SEARCH_RADIUS_M = 30
 
# OSM amenity tags that count as "a restaurant"
FOOD_AMENITY_TAGS = {
    "restaurant",
    "cafe",
    "bar",
    "fast_food",
    "food_court",
    "biergarten",
    "pub",
}

# Nominatim requires a descriptive User-Agent
HEADERS = {"User-Agent": "rat-restaurant-checker/1.0 (conorfarrell964@gmail.com)"}

OVERPASS_URL  = "https://overpass-api.de/api/interpreter"

def query_and_return(lat,lon, radius):

    # Build query with restaurant tags

    amenity_regex = "|".join(FOOD_AMENITY_TAGS)
    osm_query =  f"""
        [out:json][timeout:10];
        (
        node["amenity"~"{amenity_regex}"](around:{radius},{lat},{lon});
        way ["amenity"~"{amenity_regex}"](around:{radius},{lat},{lon});
        rel ["amenity"~"{amenity_regex}"](around:{radius},{lat},{lon});
        );
        out center;
    """

    # Run Query
    response = requests.post(OVERPASS_URL, data={"data": osm_query}, headers = HEADERS, timeout=15)
    response.raise_for_status()
    return response.json().get("elements", [])
    # return response

res = []
for _ ,row in df.iterrows():
    try:
        lat = row['latitude']
        lon = row['longitude']
        print(lat,lon)
        print(query_and_return(lat,lon,radius = SEARCH_RADIUS_M))
    except Exception as e:
        print(e)
    time.sleep(2)

    

40.727378256282 -73.982934413601
504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
40.686179132829 -73.931267975556
[]
40.707470300369 -73.801622270704
[]
40.611950625707 -73.974636942072
429 Client Error: Too Many Requests for url: https://overpass-api.de/api/interpreter


In [18]:
res

[[]]

In [16]:
tags = res.get("tags", {})
address_parts = [
    tags.get("addr:housenumber"),
    tags.get("addr:street"),
    tags.get("addr:city"),
    tags.get("addr:state"),
    tags.get("addr:postcode"),
    tags.get("addr:country"),
]
address = ", ".join(p for p in address_parts if p) or None

address

AttributeError: 'list' object has no attribute 'get'

In [26]:
"""
Restaurant Checker using Overpass API + Nominatim
--------------------------------------------------
Given an address string, determines whether it is a restaurant (or similar
food-service venue) by:
  1. Geocoding the address to coordinates via Nominatim
  2. Querying the Overpass API for food-service amenities nearby
"""

import time
import requests

# ── Constants ────────────────────────────────────────────────────────────────

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
OVERPASS_URL  = "https://overpass-api.com/api/interpreter"

# Radius in metres to search around the geocoded point
SEARCH_RADIUS_M = 30

# OSM amenity tags that count as "a restaurant"
FOOD_AMENITY_TAGS = {
    "restaurant",
    "cafe",
    "bar",
    "fast_food",
    "food_court",
    "biergarten",
    "pub",
}

# Nominatim requires a descriptive User-Agent
HEADERS = {"User-Agent": "restaurant-checker/1.0 (your@email.com)"}


# ── Step 1: Geocode ───────────────────────────────────────────────────────────

def geocode_address(address: str) -> tuple[float, float] | None:
    """
    Convert an address string to (lat, lon) using Nominatim.
    Returns None if the address could not be geocoded.
    """
    params = {
        "q": address,
        "format": "json",
        "limit": 1,
    }
    response = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
    response.raise_for_status()

    results = response.json()
    if not results:
        return None

    lat = float(results[0]["lat"])
    lon = float(results[0]["lon"])
    return lat, lon


# ── Step 2: Overpass query ────────────────────────────────────────────────────

def build_overpass_query(lat: float, lon: float, radius: int = SEARCH_RADIUS_M) -> str:
    """Build an Overpass QL query that searches for food amenities near a point."""
    amenity_regex = "|".join(FOOD_AMENITY_TAGS)
    return f"""
[out:json][timeout:10];
(
  node["amenity"~"{amenity_regex}"](around:{radius},{lat},{lon});
  way ["amenity"~"{amenity_regex}"](around:{radius},{lat},{lon});
  rel ["amenity"~"{amenity_regex}"](around:{radius},{lat},{lon});
);
out center;
"""


def query_overpass(lat: float, lon: float) -> list[dict]:
    """
    Run the Overpass query and return a list of matching OSM elements.
    Each element is a dict with at minimum 'type', 'id', and 'tags'.
    """
    query = build_overpass_query(lat, lon)
    response = requests.post(OVERPASS_URL, data={"data": query}, timeout=15)
    response.raise_for_status()
    return response.json().get("elements", [])


# ── Step 3: Decide ────────────────────────────────────────────────────────────

def is_restaurant(address: str) -> dict:
    """
    Main function. Given an address string, returns a result dict:
      {
        "address":       str,
        "is_restaurant": bool,
        "matched":       list[dict],   # OSM elements that matched
        "lat":           float | None,
        "lon":           float | None,
        "error":         str | None,
      }
    """
    result = {
        "address": address,
        "is_restaurant": False,
        "matched": [],
        "lat": None,
        "lon": None,
        "error": None,
    }

    # 1. Geocode
    coords = geocode_address(address)
    if coords is None:
        result["error"] = "Could not geocode address"
        return result

    lat, lon = coords
    result["lat"], result["lon"] = lat, lon

    # 2. Query Overpass
    elements = query_overpass(lat, lon)

    # 3. Filter to food amenities
    matches = [
        el for el in elements
        if el.get("tags", {}).get("amenity") in FOOD_AMENITY_TAGS
    ]

    result["is_restaurant"] = len(matches) > 0

    for el in matches:
        # nodes have lat/lon directly; ways/relations use the "center" key
        if el["type"] == "node":
            el_lat, el_lon = el["lat"], el["lon"]
        else:
            el_lat = el.get("center", {}).get("lat")
            el_lon = el.get("center", {}).get("lon")

        # Pull address fields directly from OSM tags — no extra API call needed.
        # OSM stores address parts separately, so we assemble them here.
        tags = el.get("tags", {})
        address_parts = [
            tags.get("addr:housenumber"),
            tags.get("addr:street"),
            tags.get("addr:city"),
            tags.get("addr:state"),
            tags.get("addr:postcode"),
            tags.get("addr:country"),
        ]
        address = ", ".join(p for p in address_parts if p) or None

        result["matched"].append({
            "osm_type": el["type"],
            "osm_id":   el["id"],
            "name":     tags.get("name", "unnamed"),
            "amenity":  tags.get("amenity"),
            "lat":      el_lat,
            "lon":      el_lon,
            "address":  address,
        })

    return result


# ── Example usage ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    test_addresses = [
        "11 Madison Ave, New York, NY 10010",   # Eleven Madison Park (restaurant)
        "1600 Pennsylvania Ave NW, Washington, DC",  # The White House (not a restaurant)
    ]

    for address in test_addresses:
        print(f"\nChecking: {address}")
        result = is_restaurant(address)

        if result["error"]:
            print(f"  ⚠ Error: {result['error']}")
        else:
            print(f"  Coordinates : {result['lat']:.5f}, {result['lon']:.5f}")
            print(f"  Is restaurant: {result['is_restaurant']}")
            if result["matched"]:
                for m in result["matched"]:
                    print(f"    → {m['name']} ({m['amenity']})")
                    print(f"       {m['address']}")

        # Nominatim rate limit: max 1 request/second
        time.sleep(1)


Checking: 11 Madison Ave, New York, NY 10010


ConnectionError: HTTPSConnectionPool(host='overpass-api.com', port=443): Max retries exceeded with url: /api/interpreter (Caused by NameResolutionError("HTTPSConnection(host='overpass-api.com', port=443): Failed to resolve 'overpass-api.com' ([Errno 11001] getaddrinfo failed)"))